# LayerLog Anatomy

Audits aggregate LayerLog fields, properties, and custom_methods against a live layer record.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## LayerLog Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerLog",
    "tl.LayerLog.PORTABLE_STATE_SPEC",
    "tl.LayerLog.out",
    "tl.LayerLog.out_postfunc",
    "tl.LayerLog.out_transform",
    "tl.LayerLog.autograd_saved_memory",
    "tl.LayerLog.num_autograd_saved_tensors",
    "tl.LayerLog.buffer_address",
    "tl.LayerLog.buffer_parent",
    "tl.LayerLog.saved_args",
    "tl.LayerLog.saved_kwargs",
    "tl.LayerLog.children",
    "tl.LayerLog.children_per_pass",
    "tl.LayerLog.child_ops_per_layer",
    "tl.LayerLog.co_parents",
    "tl.LayerLog.conditional_arm_children",
    "tl.LayerLog.conditional_elif_children",
    "tl.LayerLog.conditional_else_children",
    "tl.LayerLog.conditional_entry_children",
    "tl.LayerLog.conditional_then_children",
    "tl.LayerLog.conditional_branch_stack_ops",
    "tl.LayerLog.conditional_role_stacks",
    "tl.LayerLog.module",
    "tl.LayerLog.modules",
    "tl.LayerLog.grad_fn_log",
    "tl.LayerLog.capture_index",
    "tl.LayerLog.detach_saved_activations",
    "tl.LayerLog.edges_vary_across_ops",
    "tl.LayerLog.equivalent_ops",
    "tl.LayerLog.annotations",
    "tl.LayerLog.flops_backward",
    "tl.LayerLog.flops_forward",
    "tl.LayerLog.func",
    "tl.LayerLog.arg_names",
    "tl.LayerLog.code_context",
    "tl.LayerLog.func_config",
    "tl.LayerLog.is_inplace",
    "tl.LayerLog.func_name",
    "tl.LayerLog.func_rng_states",
    "tl.LayerLog.func_duration",
    "tl.LayerLog.get_children",
    "tl.LayerLog.get_parents",
    "tl.LayerLog.grad_fn_id",
    "tl.LayerLog.grad_fn_name",
    "tl.LayerLog.grad_fn",
    "tl.LayerLog.grad",
    "tl.LayerLog.has_children",
    "tl.LayerLog.has_co_parents",
    "tl.LayerLog.has_grad",
    "tl.LayerLog.has_parents",
    "tl.LayerLog.has_saved_outs",
    "tl.LayerLog.has_siblings",
    "tl.LayerLog.is_buffer",
    "tl.LayerLog.in_submodule",
    "tl.LayerLog.is_final_output",
    "tl.LayerLog.is_input",
    "tl.LayerLog.is_internal_source",
    "tl.LayerLog.is_internal_sink",
    "tl.LayerLog.is_output",
    "tl.LayerLog.is_part_of_iterable_output",
    "tl.LayerLog.is_scalar_bool",
    "tl.LayerLog.is_terminal_bool",
    "tl.LayerLog.multi_output_index",
    "tl.LayerLog.layer_label",
    "tl.LayerLog.layer_label_no_pass",
    "tl.LayerLog.layer_label_no_pass_short",
    "tl.LayerLog.layer_label_short",
    "tl.LayerLog.layer_label_w_pass",
    "tl.LayerLog.layer_label_w_pass_short",
    "tl.LayerLog.trace_index",
    "tl.LayerLog.layer_type",
    "tl.LayerLog.type_index",
    "tl.LayerLog.leaf_module_ops",
    "tl.LayerLog.lookup_keys",
    "tl.LayerLog.macs_backward",
    "tl.LayerLog.macs_forward",
    "tl.LayerLog.module_call_depth",
    "tl.LayerLog.num_args_total",
    "tl.LayerLog.num_kwargs",
    "tl.LayerLog.num_param_tensors",
    "tl.LayerLog.num_params_frozen",
    "tl.LayerLog.num_params",
    "tl.LayerLog.num_params_trainable",
    "tl.LayerLog.num_calls",
    "tl.LayerLog.num_pos_args",
    "tl.LayerLog.equivalence_class",
    "tl.LayerLog.compute_index",
    "tl.LayerLog.output_device",
    "tl.LayerLog.params",
    "tl.LayerLog.param_memory",
    "tl.LayerLog.param_memory_str",
    "tl.LayerLog.parent_arg_positions",
    "tl.LayerLog.parents",
    "tl.LayerLog.parents_per_pass",
    "tl.LayerLog._param_barcodes",
    "tl.LayerLog._param_logs",
    "tl.LayerLog.param_shapes",
    "tl.LayerLog.parent_ops_per_layer",
    "tl.LayerLog.call_labels",
    "tl.LayerLog.call_index",
    "tl.LayerLog.ops",
    "tl.LayerLog.save_grads",
    "tl.LayerLog.bool_value",
    "tl.LayerLog.show",
    "tl.LayerLog.siblings",
    "tl.LayerLog.source_trace",
    "tl.LayerLog.tensor",
    "tl.LayerLog.dtype",
    "tl.LayerLog.memory",
    "tl.LayerLog.memory_str",
    "tl.LayerLog.shape",
    "tl.LayerLog.transformed_out",
    "tl.LayerLog.transformed_out_dtype",
    "tl.LayerLog.transformed_out_memory",
    "tl.LayerLog.transformed_out_shape",
    "tl.LayerLog.transformed_grad",
    "tl.LayerLog.transformed_grad_dtype",
    "tl.LayerLog.transformed_grad_memory",
    "tl.LayerLog.transformed_grad_shape",
    "tl.LayerLog.uses_params",
]

pretty_print_fields(
    layer_log,
    [
        "layer_label",
        "layer_label_w_pass",
        "shape",
        "dtype",
        "has_parents",
        "has_children",
        "num_calls",
    ],
)
pass_records = layer_log.ops.values() if hasattr(layer_log.ops, "values") else layer_log.ops
print("pass labels", [p.layer_label_w_pass for p in pass_records])

## Identity and tensor data

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog",
    "tl.LayerLog.PORTABLE_STATE_SPEC",
    "tl.LayerLog.out",
    "tl.LayerLog.out_postfunc",
    "tl.LayerLog.out_transform",
    "tl.LayerLog.autograd_saved_memory",
    "tl.LayerLog.num_autograd_saved_tensors",
    "tl.LayerLog.buffer_address",
    "tl.LayerLog.buffer_parent",
    "tl.LayerLog.saved_args",
    "tl.LayerLog.saved_kwargs",
    "tl.LayerLog.children",
    "tl.LayerLog.children_per_pass",
    "tl.LayerLog.child_ops_per_layer",
    "tl.LayerLog.co_parents",
    "tl.LayerLog.conditional_arm_children",
    "tl.LayerLog.conditional_elif_children",
    "tl.LayerLog.conditional_else_children",
    "tl.LayerLog.conditional_entry_children",
    "tl.LayerLog.conditional_then_children",
    "tl.LayerLog.conditional_branch_stack_ops",
    "tl.LayerLog.conditional_role_stacks",
    "tl.LayerLog.module",
    "tl.LayerLog.modules",
    "tl.LayerLog.grad_fn_log",
]

audit_touch_items("Identity and tensor data", ITEMS, AUDIT_CONTEXT)

## Gradient and out data

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.capture_index",
    "tl.LayerLog.detach_saved_activations",
    "tl.LayerLog.edges_vary_across_ops",
    "tl.LayerLog.equivalent_ops",
    "tl.LayerLog.annotations",
    "tl.LayerLog.flops_backward",
    "tl.LayerLog.flops_forward",
    "tl.LayerLog.func",
    "tl.LayerLog.arg_names",
    "tl.LayerLog.code_context",
    "tl.LayerLog.func_config",
    "tl.LayerLog.is_inplace",
    "tl.LayerLog.func_name",
    "tl.LayerLog.func_rng_states",
    "tl.LayerLog.func_duration",
    "tl.LayerLog.get_children",
    "tl.LayerLog.get_parents",
    "tl.LayerLog.grad_fn_id",
    "tl.LayerLog.grad_fn_name",
    "tl.LayerLog.grad_fn",
    "tl.LayerLog.grad",
    "tl.LayerLog.has_children",
    "tl.LayerLog.has_co_parents",
    "tl.LayerLog.has_grad",
    "tl.LayerLog.has_parents",
]

audit_touch_items("Gradient and out data", ITEMS, AUDIT_CONTEXT)

## Graph relationships

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.has_saved_outs",
    "tl.LayerLog.has_siblings",
    "tl.LayerLog.is_buffer",
    "tl.LayerLog.in_submodule",
    "tl.LayerLog.is_final_output",
    "tl.LayerLog.is_input",
    "tl.LayerLog.is_internal_source",
    "tl.LayerLog.is_internal_sink",
    "tl.LayerLog.is_output",
    "tl.LayerLog.is_part_of_iterable_output",
    "tl.LayerLog.is_scalar_bool",
    "tl.LayerLog.is_terminal_bool",
    "tl.LayerLog.multi_output_index",
    "tl.LayerLog.layer_label",
    "tl.LayerLog.layer_label_no_pass",
    "tl.LayerLog.layer_label_no_pass_short",
    "tl.LayerLog.layer_label_short",
    "tl.LayerLog.layer_label_w_pass",
    "tl.LayerLog.layer_label_w_pass_short",
    "tl.LayerLog.trace_index",
    "tl.LayerLog.layer_type",
    "tl.LayerLog.type_index",
    "tl.LayerLog.leaf_module_ops",
    "tl.LayerLog.lookup_keys",
    "tl.LayerLog.macs_backward",
]

audit_touch_items("Graph relationships", ITEMS, AUDIT_CONTEXT)

## Module association

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.macs_forward",
    "tl.LayerLog.module_call_depth",
    "tl.LayerLog.num_args_total",
    "tl.LayerLog.num_kwargs",
    "tl.LayerLog.num_param_tensors",
    "tl.LayerLog.num_params_frozen",
    "tl.LayerLog.num_params",
    "tl.LayerLog.num_params_trainable",
    "tl.LayerLog.num_calls",
    "tl.LayerLog.num_pos_args",
    "tl.LayerLog.equivalence_class",
    "tl.LayerLog.compute_index",
    "tl.LayerLog.output_device",
    "tl.LayerLog.params",
    "tl.LayerLog.param_memory",
    "tl.LayerLog.param_memory_str",
    "tl.LayerLog.parent_arg_positions",
    "tl.LayerLog.parents",
    "tl.LayerLog.parents_per_pass",
    "tl.LayerLog._param_barcodes",
    "tl.LayerLog._param_logs",
    "tl.LayerLog.param_shapes",
    "tl.LayerLog.parent_ops_per_layer",
    "tl.LayerLog.call_labels",
    "tl.LayerLog.call_index",
]

audit_touch_items("Module association", ITEMS, AUDIT_CONTEXT)

## Pass aggregation

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.ops",
    "tl.LayerLog.save_grads",
    "tl.LayerLog.bool_value",
    "tl.LayerLog.show",
    "tl.LayerLog.siblings",
    "tl.LayerLog.source_trace",
    "tl.LayerLog.tensor",
    "tl.LayerLog.dtype",
    "tl.LayerLog.memory",
    "tl.LayerLog.memory_str",
    "tl.LayerLog.shape",
    "tl.LayerLog.transformed_out",
    "tl.LayerLog.transformed_out_dtype",
    "tl.LayerLog.transformed_out_memory",
    "tl.LayerLog.transformed_out_shape",
    "tl.LayerLog.transformed_grad",
    "tl.LayerLog.transformed_grad_dtype",
    "tl.LayerLog.transformed_grad_memory",
    "tl.LayerLog.transformed_grad_shape",
    "tl.LayerLog.uses_params",
]

audit_touch_items("Pass aggregation", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerLog",
    "tl.LayerLog.PORTABLE_STATE_SPEC",
    "tl.LayerLog.out",
    "tl.LayerLog.out_postfunc",
    "tl.LayerLog.out_transform",
    "tl.LayerLog.autograd_saved_memory",
    "tl.LayerLog.num_autograd_saved_tensors",
    "tl.LayerLog.buffer_address",
    "tl.LayerLog.buffer_parent",
    "tl.LayerLog.saved_args",
    "tl.LayerLog.saved_kwargs",
    "tl.LayerLog.children",
    "tl.LayerLog.children_per_pass",
    "tl.LayerLog.child_ops_per_layer",
    "tl.LayerLog.co_parents",
    "tl.LayerLog.conditional_arm_children",
    "tl.LayerLog.conditional_elif_children",
    "tl.LayerLog.conditional_else_children",
    "tl.LayerLog.conditional_entry_children",
    "tl.LayerLog.conditional_then_children",
    "tl.LayerLog.conditional_branch_stack_ops",
    "tl.LayerLog.conditional_role_stacks",
    "tl.LayerLog.module",
    "tl.LayerLog.modules",
    "tl.LayerLog.grad_fn_log",
    "tl.LayerLog.capture_index",
    "tl.LayerLog.detach_saved_activations",
    "tl.LayerLog.edges_vary_across_ops",
    "tl.LayerLog.equivalent_ops",
    "tl.LayerLog.annotations",
    "tl.LayerLog.flops_backward",
    "tl.LayerLog.flops_forward",
    "tl.LayerLog.func",
    "tl.LayerLog.arg_names",
    "tl.LayerLog.code_context",
    "tl.LayerLog.func_config",
    "tl.LayerLog.is_inplace",
    "tl.LayerLog.func_name",
    "tl.LayerLog.func_rng_states",
    "tl.LayerLog.func_duration",
    "tl.LayerLog.get_children",
    "tl.LayerLog.get_parents",
    "tl.LayerLog.grad_fn_id",
    "tl.LayerLog.grad_fn_name",
    "tl.LayerLog.grad_fn",
    "tl.LayerLog.grad",
    "tl.LayerLog.has_children",
    "tl.LayerLog.has_co_parents",
    "tl.LayerLog.has_grad",
    "tl.LayerLog.has_parents",
    "tl.LayerLog.has_saved_outs",
    "tl.LayerLog.has_siblings",
    "tl.LayerLog.is_buffer",
    "tl.LayerLog.in_submodule",
    "tl.LayerLog.is_final_output",
    "tl.LayerLog.is_input",
    "tl.LayerLog.is_internal_source",
    "tl.LayerLog.is_internal_sink",
    "tl.LayerLog.is_output",
    "tl.LayerLog.is_part_of_iterable_output",
    "tl.LayerLog.is_scalar_bool",
    "tl.LayerLog.is_terminal_bool",
    "tl.LayerLog.multi_output_index",
    "tl.LayerLog.layer_label",
    "tl.LayerLog.layer_label_no_pass",
    "tl.LayerLog.layer_label_no_pass_short",
    "tl.LayerLog.layer_label_short",
    "tl.LayerLog.layer_label_w_pass",
    "tl.LayerLog.layer_label_w_pass_short",
    "tl.LayerLog.trace_index",
    "tl.LayerLog.layer_type",
    "tl.LayerLog.type_index",
    "tl.LayerLog.leaf_module_ops",
    "tl.LayerLog.lookup_keys",
    "tl.LayerLog.macs_backward",
    "tl.LayerLog.macs_forward",
    "tl.LayerLog.module_call_depth",
    "tl.LayerLog.num_args_total",
    "tl.LayerLog.num_kwargs",
    "tl.LayerLog.num_param_tensors",
    "tl.LayerLog.num_params_frozen",
    "tl.LayerLog.num_params",
    "tl.LayerLog.num_params_trainable",
    "tl.LayerLog.num_calls",
    "tl.LayerLog.num_pos_args",
    "tl.LayerLog.equivalence_class",
    "tl.LayerLog.compute_index",
    "tl.LayerLog.output_device",
    "tl.LayerLog.params",
    "tl.LayerLog.param_memory",
    "tl.LayerLog.param_memory_str",
    "tl.LayerLog.parent_arg_positions",
    "tl.LayerLog.parents",
    "tl.LayerLog.parents_per_pass",
    "tl.LayerLog._param_barcodes",
    "tl.LayerLog._param_logs",
    "tl.LayerLog.param_shapes",
    "tl.LayerLog.parent_ops_per_layer",
    "tl.LayerLog.call_labels",
    "tl.LayerLog.call_index",
    "tl.LayerLog.ops",
    "tl.LayerLog.save_grads",
    "tl.LayerLog.bool_value",
    "tl.LayerLog.show",
    "tl.LayerLog.siblings",
    "tl.LayerLog.source_trace",
    "tl.LayerLog.tensor",
    "tl.LayerLog.dtype",
    "tl.LayerLog.memory",
    "tl.LayerLog.memory_str",
    "tl.LayerLog.shape",
    "tl.LayerLog.transformed_out",
    "tl.LayerLog.transformed_out_dtype",
    "tl.LayerLog.transformed_out_memory",
    "tl.LayerLog.transformed_out_shape",
    "tl.LayerLog.transformed_grad",
    "tl.LayerLog.transformed_grad_dtype",
    "tl.LayerLog.transformed_grad_memory",
    "tl.LayerLog.transformed_grad_shape",
    "tl.LayerLog.uses_params",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")